# ThinkingCap-Test: Colab runner
Select **Runtime → Change runtime type → GPU**. Keep this notebook thin; the Git repository is the source of truth.

In [ ]:
import os, subprocess, sys, torch
print(torch.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO CUDA GPU')
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing'

In [ ]:
REPO_URL = 'https://github.com/BottleCapAI/ThinkingCap-Test.git'  # change to your fork
REPO_REF = 'main'  # preferably an exact commit or experiment branch
FULL_EVALUATION = False
CHECKOUT = '/content/ThinkingCap-Test'

In [ ]:
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, CHECKOUT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{CHECKOUT}/requirements.txt'], check=True)

In [ ]:
runner = os.path.join(CHECKOUT, 'cloud/run_eval.py')
if os.path.exists(runner):
    command = [sys.executable, 'cloud/run_eval.py', '--out', '/content/runs/base', '--dtype', 'auto']
    if FULL_EVALUATION:
        command.append('--full')
    subprocess.run(command, cwd=CHECKOUT, check=True)
else:
    # Compatibility fallback when cloning the untouched upstream repository.
    path = os.path.join(CHECKOUT, 'evaluate.py')
    source = open(path).read()
    needle = 'tokenizer_revision=arguments.revision,\n'
    assert needle in source
    open(path, 'w').write(source.replace(needle, needle + '                dtype=\"half\",\n', 1))
    subprocess.run([sys.executable, 'utils/eval/benchmarks.py'], cwd=CHECKOUT, check=True)
    command = [sys.executable, 'evaluate.py', '--out', '/content/runs/base']
    if not FULL_EVALUATION:
        command += ['--smoke', '--samples', '1']
    subprocess.run(command, cwd=CHECKOUT, check=True)

In [ ]:
!find /content/runs -maxdepth 3 -type f -ls
!tar -czf /content/thinkingcap-results.tgz -C /content runs
print('Download /content/thinkingcap-results.tgz before releasing the runtime.')